## Import packages

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import matplotlib.pyplot as plt
from src import load_dataset, get_class_weights, create_validation_split
from src import create_data_generators

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black'
})

## Define paths

In [ ]:
data_dir = '../data/chest_xray'
print(f"Data directory: {data_dir}")
print(f"Exists: {os.path.exists(data_dir)}")

## Create proper validation split

In [ ]:
# The Kaggle dataset has only 16 validation images — too few
# Move 15% of training data to validation
create_validation_split(data_dir, val_ratio=0.15)

## Load dataset without augmentation (baseline)

In [ ]:
train_gen, val_gen, test_gen = load_dataset(
    data_dir,
    target_size=(224, 224),
    batch_size=32
)

## Calculate class weights

In [ ]:
class_weights = get_class_weights(train_gen)

## Create augmented data generators

In [ ]:
train_gen_aug, val_gen_aug, test_gen_aug = create_data_generators(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    augment=True
)

## Visualize original vs augmented images

In [ ]:
# Get a batch of original images
train_gen_orig, _, _ = create_data_generators(data_dir, augment=False)
orig_batch = next(train_gen_orig)

# Get a batch of augmented images
aug_batch = next(train_gen_aug)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Original (top) vs Augmented (bottom)', fontsize=14)

for i in range(4):
    axes[0, i].imshow(orig_batch[0][i])
    axes[0, i].set_title('Original', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(aug_batch[0][i])
    axes[1, i].set_title('Augmented', fontsize=10)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## Visualize augmentation variations of a single image

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

# Load a single image
sample_dir = os.path.join(data_dir, 'train', 'PNEUMONIA')
sample_img_name = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpeg', '.jpg', '.png'))][0]
sample_img = load_img(os.path.join(sample_dir, sample_img_name), target_size=(224, 224))
sample_array = img_to_array(sample_img) / 255.0
sample_batch = np.expand_dims(sample_array, axis=0)

# Create augmentation generator
aug_gen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Same Image with Different Augmentations', fontsize=14)

axes[0, 0].imshow(sample_array)
axes[0, 0].set_title('Original', fontsize=10)
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flatten()[1:], 1):
    aug_img = next(aug_gen.flow(sample_batch))[0]
    ax.imshow(aug_img)
    ax.set_title(f'Augmented #{i}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Verify image shapes and pixel ranges

In [ ]:
batch_images, batch_labels = next(train_gen_aug)

print(f"Batch shape: {batch_images.shape}")
print(f"Label shape: {batch_labels.shape}")
print(f"Pixel range: [{batch_images.min():.3f}, {batch_images.max():.3f}]")
print(f"Labels in batch: {np.unique(batch_labels)}")
print(f"Label distribution: Normal={np.sum(batch_labels == 0)}, Pneumonia={np.sum(batch_labels == 1)}")

## Save preprocessing config

In [ ]:
import json

config = {
    "target_size": [224, 224],
    "batch_size": 32,
    "augmentation": True,
    "val_ratio": 0.15,
    "class_weights": {str(k): v for k, v in class_weights.items()},
    "train_samples": train_gen_aug.samples,
    "val_samples": val_gen_aug.samples,
    "test_samples": test_gen_aug.samples
}

os.makedirs('../models', exist_ok=True)
with open('../models/preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Preprocessing config saved to ../models/preprocessing_config.json")


## Summary

In [ ]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"Image size: 224 x 224")
print(f"Batch size: 32")
print(f"Training samples: {train_gen_aug.samples}")
print(f"Validation samples: {val_gen_aug.samples}")
print(f"Test samples: {test_gen_aug.samples}")
print(f"Augmentation: Enabled")
print(f"Class weights: {class_weights}")
print("=" * 50)
print("\nNext: 03_baseline_model.ipynb")